# 🚗 ZALO AI Traffic QA - Fine-tuning Pipeline

**Complete pipeline for fine-tuning Vision Language Models on Vietnamese Traffic Videos**

## Pipeline Overview:
- **Part 1**: Data Preparation & Preprocessing (Parallel frame extraction + caching)
- **Part 2**: Model Fine-tuning with SFT (Supervised Fine-Tuning with LoRA)

**Optimizations:**
- ⚡ Parallel preprocessing with multiprocessing
- 💾 Frame caching for faster re-runs
- 🎯 LoRA for efficient fine-tuning
- 📊 Progress tracking and validation

---

# PART 1: DATA PREPARATION

This section handles:
1. Mounting Google Drive and extracting data
2. Parallel frame extraction from videos
3. Caching preprocessed frames
4. Creating training dataset with proper structure
5. Testing model inference before fine-tuning

## 📁 Step 1.1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install -q opencv-python tqdm

In [4]:
# import zipfile
# import os
# from tqdm import tqdm

# # Assuming your zip file is in Google Drive and mounted at /content/drive/MyDrive/
# zip_file_path = "/content/drive/MyDrive/ZALO_DRIVE/traffic_buddy_train.zip"
# extraction_path = '/content/drive/MyDrive/ZALO_DRIVE/' # Unzip to local Colab disk

# os.makedirs(extraction_path, exist_ok=True) # Create the directory if it doesn't exist

# with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#     # Get list of members for tqdm
#     members = zip_ref.namelist()
#     with tqdm(total=len(members), desc="Extracting files") as pbar:
#         for member in members:
#             zip_ref.extract(member, extraction_path)
#             pbar.update(1)

Extracting files: 100%|██████████| 745/745 [03:42<00:00,  3.35it/s]


## ⚙️ Step 1.4: Configuration and Paths

In [4]:
import os

# ============================================================================
# PATHS - Google Drive
# ============================================================================
DATA_ROOT = "/content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test"
TRAIN_JSON = f"{DATA_ROOT}/train/train.json"
TRAIN_VIDEOS = f"{DATA_ROOT}/train/videos"

# Cache directory on Google Drive
CACHE_DIR = "/content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache"
FRAMES_CACHE_DIR = os.path.join(CACHE_DIR, "extracted_frames")
MODEL_CACHE_DIR = os.path.join(CACHE_DIR, "models")

# Create directories
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FRAMES_CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# ============================================================================
# PREPROCESSING SETTINGS
# ============================================================================
NUM_WORKERS = 24  # Parallel workers for frame extraction (adjust based on CPU cores)
MAX_FRAMES_PER_VIDEO = 8  # Maximum frames to extract per video

# ============================================================================
# MODEL SETTINGS
# ============================================================================
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# ============================================================================
# TRAINING SETTINGS
# ============================================================================
EPOCHS = 3
BATCH_SIZE = 1
LEARNING_RATE = 2e-4
LOGGING_STEPS = 50
EVAL_STEPS = 100
SAVE_STEPS = 100
TRAIN_TEST_SPLIT = 0.1  # 10% for evaluation

print("✅ Configuration loaded")
print(f"📁 Data root: {DATA_ROOT}")
print(f"📁 Train JSON: {TRAIN_JSON}")
print(f"📁 Train videos: {TRAIN_VIDEOS}")
print(f"💾 Cache dir: {CACHE_DIR}")

✅ Configuration loaded
📁 Data root: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test
📁 Train JSON: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/train.json
📁 Train videos: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos
💾 Cache dir: /content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache


In [6]:
import os

print("\nChecking if data directories exist...")

if not os.path.exists(DATA_ROOT):
    print(f"❌ Error: DATA_ROOT not found at {DATA_ROOT}")
else:
    print(f"✅ DATA_ROOT found at {DATA_ROOT}")

if not os.path.exists(TRAIN_JSON):
    print(f"❌ Error: TRAIN_JSON not found at {TRAIN_JSON}")
else:
    print(f"✅ TRAIN_JSON found at {TRAIN_JSON}")

if not os.path.exists(TRAIN_VIDEOS):
    print(f"❌ Error: TRAIN_VIDEOS not found at {TRAIN_VIDEOS}")
else:
    print(f"✅ TRAIN_VIDEOS found at {TRAIN_VIDEOS}")

if not os.path.exists(CACHE_DIR):
    print(f"❌ Error: CACHE_DIR not found at {CACHE_DIR}")
else:
    print(f"✅ CACHE_DIR found at {CACHE_DIR}")

if not os.path.exists(FRAMES_CACHE_DIR):
    print(f"❌ Error: FRAMES_CACHE_DIR not found at {FRAMES_CACHE_DIR}")
else:
    print(f"✅ FRAMES_CACHE_DIR found at {FRAMES_CACHE_DIR}")

if not os.path.exists(MODEL_CACHE_DIR):
    print(f"❌ Error: MODEL_CACHE_DIR not found at {MODEL_CACHE_DIR}")
else:
    print(f"✅ MODEL_CACHE_DIR found at {MODEL_CACHE_DIR}")


Checking if data directories exist...
✅ DATA_ROOT found at /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test
✅ TRAIN_JSON found at /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/train.json
✅ TRAIN_VIDEOS found at /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos
✅ CACHE_DIR found at /content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache
✅ FRAMES_CACHE_DIR found at /content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache/extracted_frames
✅ MODEL_CACHE_DIR found at /content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache/models


In [7]:
!lscpu | grep 'CPU(s):'
!lscpu | grep 'Core(s) per socket:'
!lscpu | grep 'Thread(s) per core:'

CPU(s):                                  24
NUMA node0 CPU(s):                       0-23
Core(s) per socket:                      12
Thread(s) per core:                      2


In [ ]:
!lscpu

## 🛠️ Step 1.5: Utility Functions

In [11]:
!pip install -q decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 14.2 MB/s eta 0:00:00


In [8]:
import json
import cv2
from PIL import Image
import hashlib
import pickle
from decord import VideoReader, cpu

def load_json_data(json_path):
    """Load data from JSON file"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Loaded {data['__count__']} questions from {json_path}")
    return data

def resolve_video_path(json_video_path):
    """Find absolute path of video"""
    abs_path = os.path.join(DATA_ROOT, json_video_path)
    if os.path.exists(abs_path):
        return abs_path
    raise FileNotFoundError(f"Video not found: {abs_path}")

def extract_frames_hybrid(video_path, support_frames=None, max_frames=MAX_FRAMES_PER_VIDEO):
    """
    Hybrid approach: Extract at support_frames first, then fill with uniform sampling

    Args:
        video_path: Path to video file
        support_frames: List of important timestamps (optional)
        max_frames: Maximum number of frames to extract

    Returns:
        extracted_frames: List of PIL Images
        frame_timestamps: List of timestamps
    """
    try:
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        fps = vr.get_avg_fps()

        extracted_indices = []
        extracted_timestamps = []

        # Step 1: Extract frames at support_frames timestamps
        if support_frames and len(support_frames) > 0:
            for timestamp in support_frames[:max_frames]:
                frame_idx = int(timestamp * fps)
                if 0 <= frame_idx < total_frames:
                    extracted_indices.append(frame_idx)
                    extracted_timestamps.append(timestamp)

        # Step 2: Fill remaining slots with uniform sampling
        remaining_slots = max_frames - len(extracted_indices)
        if remaining_slots > 0:
            uniform_indices = np.linspace(0, total_frames - 1, num=max_frames, dtype=int)

            for idx in uniform_indices:
                if len(extracted_indices) >= max_frames:
                    break
                # Avoid duplicates
                if idx not in extracted_indices:
                    extracted_indices.append(idx)
                    ts = vr.get_frame_timestamp(idx)
                    extracted_timestamps.append(ts[0])

        # Sort by frame index
        sorted_pairs = sorted(zip(extracted_indices, extracted_timestamps))
        extracted_indices = [p[0] for p in sorted_pairs]
        extracted_timestamps = [p[1] for p in sorted_pairs]

        # Extract frames
        frames_array = vr.get_batch(extracted_indices).asnumpy()
        extracted_frames = [Image.fromarray(frame) for frame in frames_array]

        return extracted_frames, extracted_timestamps

    except Exception as e:
        print(f"⚠️ Error in hybrid extraction: {e}")
        return [], []

print("✅ Hybrid frame extraction function loaded")

def get_cache_path(video_path, support_frames):
    """Generate unique cache path for extracted frames"""
    hash_input = f"{video_path}_{support_frames}".encode()
    cache_id = hashlib.md5(hash_input).hexdigest()
    cache_file = os.path.join(FRAMES_CACHE_DIR, f"{cache_id}.pkl")
    return cache_file

print("✅ Utility functions loaded")

✅ Hybrid frame extraction function loaded
✅ Utility functions loaded


## 🚀 Step 1.6: Parallel Preprocessing Pipeline

Extract frames from all videos in parallel and cache them for fast re-use

In [9]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np

def extract_and_cache_frames(sample_data):
    """Extract frames for one sample and cache to disk"""
    try:
        video_path = resolve_video_path(sample_data['video_path'])
        cache_file = get_cache_path(video_path, sample_data['support_frames'])

        # Check if already cached
        if os.path.exists(cache_file):
            return {
                'id': sample_data['id'],
                'status': 'cached',
                'cache_path': cache_file
            }

        # Extract frames uniformly but prioritize at specific timestamps (given in the dataset)
        frames, timestamps = extract_frames_hybrid(
            video_path,
            support_frames=sample_data['support_frames'],
            max_frames=MAX_FRAMES_PER_VIDEO
        )

        if not frames:
            return {
                'id': sample_data['id'],
                'status': 'failed',
                'error': 'No frames extracted'
            }

        # Save to cache
        with open(cache_file, 'wb') as f:
            pickle.dump({
                'frames': frames,
                'timestamps': timestamps,
                'id': sample_data['id'],
                'video_path': video_path,
                'support_frames': sample_data['support_frames']
            }, f)

        return {
            'id': sample_data['id'],
            'status': 'success',
            'cache_path': cache_file,
            'num_frames': len(frames)
        }

    except Exception as e:
        return {
            'id': sample_data['id'],
            'status': 'error',
            'error': str(e)
        }

def preprocess_dataset(data_list, num_workers=NUM_WORKERS):
    """Preprocess entire dataset in parallel"""
    print(f"\n🚀 Starting parallel preprocessing with {num_workers} workers")
    print(f"📁 Cache directory: {FRAMES_CACHE_DIR}")
    print(f"📊 Total samples: {len(data_list)}\n")

    results = []

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(extract_and_cache_frames, sample): sample
            for sample in data_list
        }

        with tqdm(total=len(data_list), desc="Extracting frames") as pbar:
            for future in as_completed(futures):
                result = future.result()
                results.append(result)
                pbar.update(1)

                if result['status'] == 'error':
                    pbar.write(f"❌ Error {result['id']}: {result.get('error', 'Unknown')}")

    # Summary
    success_count = sum(1 for r in results if r['status'] in ['success', 'cached'])
    cached_count = sum(1 for r in results if r['status'] == 'cached')
    error_count = sum(1 for r in results if r['status'] == 'error')

    print(f"\n{'='*60}")
    print(f"📈 PREPROCESSING SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Total successful: {success_count}/{len(data_list)}")
    print(f"💾 Already cached: {cached_count}")
    print(f"🆕 Newly processed: {success_count - cached_count}")
    print(f"❌ Errors: {error_count}")
    print(f"{'='*60}\n")

    return results

def create_index_file(results, output_path):
    """Create index file mapping question IDs to cache paths"""
    index = {
        r['id']: r['cache_path']
        for r in results
        if r['status'] in ['success', 'cached']
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(index, f, ensure_ascii=False, indent=2)

    print(f"✅ Created index file: {output_path}")
    return index

print("✅ Preprocessing functions loaded")

✅ Preprocessing functions loaded


## 🎬 Step 1.7: Run Preprocessing

This will extract and cache all video frames in parallel

In [ ]:
# Load training data
train_data = load_json_data(TRAIN_JSON)

# Process all samples (or use [:10] for testing)
data_to_process = train_data['data']  # All samples
# data_to_process = train_data['data'][:10]  # Test with 10 samples

# Run preprocessing
preprocessing_results = preprocess_dataset(data_to_process)

# Create index file - WARNING: can take a lot of space in Drive
# index_path = os.path.join(CACHE_DIR, "frames_index.json")
# frames_index = create_index_file(preprocessing_results, index_path) # create index for cache file

print(f"\n✅ Preprocessing complete!")
print(f"📁 Frames cached in: {FRAMES_CACHE_DIR}")
print(f"📄 Index file: {index_path}")
# ❌ Error train_0027: extract_frames_hybrid() got an unexpected keyword argument 'num_frames'

✅ Loaded 1490 questions from /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/train.json

🚀 Starting parallel preprocessing with 24 workers
📁 Cache directory: /content/drive/MyDrive/ZALO_DRIVE/zalo_finetune_cache/extracted_frames
📊 Total samples: 1490



Extracting frames:  47%|████▋     | 701/1490 [25:00<1:30:34,  6.89s/it]

## 📊 Step 1.8: Create Dataset with Proper Structure

Create a pandas DataFrame with all fields: id, question, choices, answer, support_frames, video_path

In [15]:
import pandas as pd
from IPython.display import display

# Create dataset table
dataset_records = []

for sample in train_data['data']:
    dataset_records.append({
        'id': sample['id'],
        'question': sample['question'],
        'choices': sample['choices'],
        'support_frames': sample['support_frames'],
        'video_path': sample['video_path']
    })

# Create DataFrame
df_dataset = pd.DataFrame(dataset_records)

print(f"✅ Created dataset with {len(df_dataset)} samples")
print(f"\n📊 Dataset Structure:")
display(df_dataset.head())
print(f"\n📈 Dataset Info:")
print(df_dataset.info())

NameError: name 'train_data' is not defined

## 🧪 Step 1.9: Test Model Inference (Before Fine-tuning)

Test the pre-trained model on a sample video to verify it works correctly using Structured Video Captioning approach

In [ ]:
%pip install -q transformers qwen-vl-utils pillow accelerate
%pip install -q flash-attn --no-build-isolation
%pip install -q peft bitsandbytes scipy

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print(torch.cuda.is_available())

# Load model for testing
print("\n🔧 Loading model for inference test...")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"✅ Model loaded on: {model.device}")
print(f"✅ Dtype: {model.dtype}")

True

🔧 Loading model for inference test...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.91G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
def inference_video(video_path, prompt, max_new_tokens=2048, total_pixels=20480 * 28 * 28, min_pixels=16 * 28 * 28):
    """
    Run inference on a video

    Args:
        video_path: Path to video file
        prompt: Text prompt for the model
        max_new_tokens: Maximum tokens to generate
        total_pixels: Total pixel budget for video processing
        min_pixels: Minimum pixels per frame


    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"video": video_path,
                 "total_pixels": total_pixels,
                 "min_pixels": min_pixels,
                #  "max_pixels": 128*32*32,
                #  "max_frames": 16,
                 },
            ]
        },
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info([messages], return_video_kwargs=True)
    fps_inputs = video_kwargs['fps'][0]
    print('fps_inputs:', video_kwargs['fps'])

    print(f"📹 Video input shape: {video_inputs[0].shape}")
    num_frames, _, resized_height, resized_width = video_inputs[0].shape
    print(f"🎬 Number of frames: {num_frames}")
    print(f"📊 Number of video tokens: {int(num_frames / 2 * resized_height / 28 * resized_width / 28)}")

    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, fps=fps_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    return output_text[0]

print("✅ Inference function loaded")


In [ ]:
import os
import hashlib
import requests

from IPython.display import Markdown, display
import numpy as np
from PIL import Image
import decord
from decord import VideoReader, cpu


def download_video(url, dest_path):
    response = requests.get(url, stream=True)
    with open(dest_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8096):
            f.write(chunk)
    print(f"Video downloaded to {dest_path}")


def get_video_frames(video_path, num_frames=128, cache_dir='.cache'):
    os.makedirs(cache_dir, exist_ok=True)

    video_hash = hashlib.md5(video_path.encode('utf-8')).hexdigest()
    if video_path.startswith('http://') or video_path.startswith('https://'):
        video_file_path = os.path.join(cache_dir, f'{video_hash}.mp4')
        if not os.path.exists(video_file_path):
            download_video(video_path, video_file_path)
    else:
        video_file_path = video_path

    frames_cache_file = os.path.join(cache_dir, f'{video_hash}_{num_frames}_frames.npy')
    timestamps_cache_file = os.path.join(cache_dir, f'{video_hash}_{num_frames}_timestamps.npy')

    if os.path.exists(frames_cache_file) and os.path.exists(timestamps_cache_file):
        frames = np.load(frames_cache_file)
        timestamps = np.load(timestamps_cache_file)
        return video_file_path, frames, timestamps

    vr = VideoReader(video_file_path, ctx=cpu(0))
    total_frames = len(vr)

    indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
    frames = vr.get_batch(indices).asnumpy()
    timestamps = np.array([vr.get_frame_timestamp(idx) for idx in indices])

    np.save(frames_cache_file, frames)
    np.save(timestamps_cache_file, timestamps)

    return video_file_path, frames, timestamps


def create_image_grid(images, num_columns=8):
    pil_images = [Image.fromarray(image) for image in images]
    num_rows = (len(images) + num_columns - 1) // num_columns

    img_width, img_height = pil_images[0].size
    grid_width = num_columns * img_width
    grid_height = num_rows * img_height
    grid_image = Image.new('RGB', (grid_width, grid_height))

    for idx, image in enumerate(pil_images):
        row_idx = idx // num_columns
        col_idx = idx % num_columns
        position = (col_idx * img_width, row_idx * img_height)
        grid_image.paste(image, position)

    return grid_image

In [ ]:
# Structured video captioning prompt (similar to video_understanding_simple.ipynb)
# def inference(video_path, sample):
#   prompt = """
#   Localize a series of activity events in the video, output the start and end timestamp for each event, and describe each event with sentences.
#   Provide the result in json format with 'mm:ss.ff' format for time depiction.

#   Additionally, answer this question about the video:
#   Question:{question}
#   Choices:{choices}

#   Provide your answer in Vietnamese.""".format(
#       question=sample['question'],
#       choices='\n'.join(sample['choices'])
#   )
#   print('Full Prompt:', prompt)
#   print(f"\n{'='*80}")
#   print(f"🧪 TESTING INFERENCE ON VIDEO: {sample['id']}")
#   print(f"📹 Video: {video_path}")
#   print(f"❓ Question: {sample['question']}")
#   print(f"✅ Answer: {sample['answer']}")

#   response = inference_video(video_path, prompt)
#   print(f"📄 MODEL RESPONSE:")
#   print(response)

#   print("✅ Inference test complete! Model is working correctly.")

prompt = """
  Localize a series of activity events in the video, output the start and end timestamp for each event, and describe each event with sentences.
  Provide the result in json format with 'mm:ss.ff' format for time depiction.

  Additionally, answer this question about the video:
  Question:{question}
  Choices:{choices}

  Provide your answer in Vietnamese.""".format(
      question=sample['question'],
      choices='\n'.join(sample['choices'])
  )

def inference(video_path, prompt, max_new_tokens=2048, total_pixels=20480 * 28 * 28, min_pixels=16 * 28 * 28):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"video": video_path, "total_pixels": total_pixels, "min_pixels": min_pixels},
            ]
        },
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs, video_kwargs = process_vision_info([messages], return_video_kwargs=True)
    fps_inputs = video_kwargs['fps'][0]
    print("video input:", video_inputs[0].shape)
    num_frames, _, resized_height, resized_width = video_inputs[0].shape
    print("num of video tokens:", int(num_frames / 2 * resized_height / 28 * resized_width / 28))
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, fps=fps_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to('cuda')

    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return output_text[0]

N = 1
samples = train_data['data'][:N]
# # Play the video
# display(Video(video_path, embed=True, width=800, height=600))
for sample in samples:
  video_path = resolve_video_path(sample['video_path'])
  inference(video_path, sample)

In [ ]:
# # Clean up model from memory before fine-tuning
del model
del processor
torch.cuda.empty_cache()

print("✅ Cleared model from memory")

---

# PART 2: FINE-TUNING WITH SFT

This section handles:
1. Loading preprocessed data from cache
2. Creating training dataset with proper formatting
3. Setting up LoRA configuration for efficient fine-tuning
4. Training the model with SFTTrainer
5. Saving and evaluating the fine-tuned model

## 📚 Step 2.1: Create Training Dataset from Cached Frames

In [ ]:
from torch.utils.data import Dataset

class ZaloTrafficDataset(Dataset):
    """Dataset that loads preprocessed frames from cache"""

    def __init__(self, data_list, frames_index):
        """
        Args:
            data_list: List of data samples from train.json
            frames_index: Dictionary mapping question IDs to cache paths
        """
        self.data_list = data_list
        self.frames_index = frames_index

        # Filter out samples without cached frames
        self.valid_samples = [
            s for s in data_list
            if s['id'] in frames_index
        ]

        print(f"✅ Dataset created with {len(self.valid_samples)}/{len(data_list)} valid samples")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        sample = self.valid_samples[idx]

        # Load cached frames
        cache_path = self.frames_index[sample['id']]
        with open(cache_path, 'rb') as f:
            cached_data = pickle.load(f)

        frames = cached_data['frames']

        # Format question with choices
        question_text = f"{sample['question']}\n"
        for choice in sample['choices']:
            question_text += f"{choice}\n"
        question_text = question_text.strip()

        return {
            'id': sample['id'],
            'frames': frames,  # List of PIL Images
            'question': question_text,
            'answer': sample['answer'],
            'video_path': cached_data['video_path'],
            'support_frames': cached_data['support_frames'],
            'timestamps': cached_data['timestamps']
        }

print("✅ ZaloTrafficDataset class defined")

In [ ]:
# System message for Vietnamese traffic analysis
SYSTEM_MESSAGE = """Bạn là một trợ lý AI chuyên phân tích video giao thông tại Việt Nam.
Nhiệm vụ của bạn là trả lời các câu hỏi về biển báo, vạch kẻ đường, và quy tắc giao thông dựa trên video."""

def format_sample_for_training(sample):
    """
    Format a sample for training with SFT

    Args:
        sample: Dictionary with 'frames', 'question', 'answer'

    Returns:
        List of messages in chat format
    """
    frames = sample['frames']  # List of PIL Images

    # Build content with all frames
    content = []
    for frame in frames:
        content.append({
            "type": "image",
            "image": frame
        })

    # Add question text
    content.append({
        "type": "text",
        "text": sample['question']
    })

    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_MESSAGE}]
        },
        {
            "role": "user",
            "content": content
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample['answer']}]
        }
    ]

print("✅ Training data formatting function defined")

In [ ]:
from sklearn.model_selection import train_test_split

# Load frames index
with open(index_path, 'r', encoding='utf-8') as f:
    frames_index = json.load(f)

# Create full dataset
full_dataset = ZaloTrafficDataset(train_data['data'], frames_index)

# Split into train/eval
indices = list(range(len(full_dataset)))
train_indices, eval_indices = train_test_split(
    indices,
    test_size=TRAIN_TEST_SPLIT,
    random_state=42
)

# Create formatted datasets for training
train_dataset = [
    format_sample_for_training(full_dataset[i])
    for i in train_indices
]

eval_dataset = [
    format_sample_for_training(full_dataset[i])
    for i in eval_indices
]

print(f"\n✅ Dataset split complete:")
print(f"📊 Training samples: {len(train_dataset)}")
print(f"📊 Evaluation samples: {len(eval_dataset)}")
print(f"\n📝 Sample formatted data (first training example):")
print(f"   - System message: {train_dataset[0][0]['content'][0]['text'][:50]}...")
print(f"   - Number of images: {sum(1 for item in train_dataset[0][1]['content'] if item['type'] == 'image')}")
print(f"   - Question preview: {train_dataset[0][1]['content'][-1]['text'][:100]}...")
print(f"   - Answer: {train_dataset[0][2]['content'][0]['text']}")

## 🔧 Step 2.2: Load Model with LoRA Configuration

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
from qwen_vl_utils import process_vision_info

print("\n🔧 Loading model with LoRA for fine-tuning...")

# Quantization config for memory efficiency
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

# Load model
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
    # quantization_config=bnb_config,
    # trust_remote_code=True,
)


# Load processor
processor = AutoProcessor.from_pretrained(MODEL_ID)

# LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,  # Alpha scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded with LoRA")
print(f"📊 Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"📊 Total parameters: {total_params:,}")
print(f"🎯 Device: {model.device}")

## 🎯 Step 2.3: Setup Training Configuration

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Calculate total training steps
NUM_STEPS = (len(train_dataset) // BATCH_SIZE) * EPOCHS

# Training arguments
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_CACHE_DIR, "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,  # Effective batch size = 4
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=LOGGING_STEPS,
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy="steps",
    bf16=True,  # Use bfloat16 for training
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",  # Disable wandb/tensorboard
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print(f"\n✅ Training configuration:")
print(f"📊 Total training steps: {NUM_STEPS}")
print(f"📊 Epochs: {EPOCHS}")
print(f"📊 Batch size: {BATCH_SIZE}")
print(f"📊 Gradient accumulation: 4 (effective batch size: {BATCH_SIZE * 4})")
print(f"📊 Learning rate: {LEARNING_RATE}")
print(f"📊 Total training samples: {len(train_dataset)}")
print(f"📊 Total evaluation samples: {len(eval_dataset)}")

## 🔄 Step 2.4: Define Data Collator

In [ ]:
def collate_fn(examples):
    """
    Collate function that handles multiple images per sample

    Args:
        examples: List of training examples in chat format

    Returns:
        Batch dictionary with input_ids, attention_mask, pixel_values, and labels
    """
    texts = []
    all_images = []

    for example in examples:
        # Apply chat template
        text = processor.apply_chat_template(example, tokenize=False, add_generation_prompt=False)
        texts.append(text)

        # Extract all images from user content
        images = [
            item["image"]
            for item in example[1]["content"]
            if item["type"] == "image"
        ]
        all_images.extend(images)

    # Process batch
    batch = processor(
        text=texts,
        images=all_images if all_images else None,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    )

    # Set labels (mask padding tokens)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels

    return batch

print("✅ Collate function defined")

## 🚀 Step 2.5: Create Trainer and Start Fine-tuning

In [ ]:
# Create SFT Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    peft_config=lora_config,
    processing_class=processor.tokenizer,
)

print("\n✅ Trainer created successfully")
print("\n" + "="*80)
print("🚀 STARTING FINE-TUNING")
print("="*80 + "\n")

In [ ]:
# Start training
train_result = trainer.train()

print("\n" + "="*80)
print("✅ TRAINING COMPLETE!")
print("="*80 + "\n")

# Print training metrics
print("📊 Training Metrics:")
for key, value in train_result.metrics.items():
    print(f"   {key}: {value}")

## 💾 Step 2.6: Save Fine-tuned Model

In [ ]:
# Save the fine-tuned model
output_model_dir = os.path.join(MODEL_CACHE_DIR, "zalo_traffic_finetuned")
trainer.save_model(output_model_dir)
processor.save_pretrained(output_model_dir)

print(f"\n✅ Model saved to: {output_model_dir}")

# Save to Google Drive for persistence
gdrive_model_dir = "/content/drive/MyDrive/ZAIC-2025/zalo_traffic_finetuned_model"
trainer.save_model(gdrive_model_dir)
processor.save_pretrained(gdrive_model_dir)

print(f"✅ Model also saved to Google Drive: {gdrive_model_dir}")

## 📈 Step 2.7: Evaluate Fine-tuned Model

In [ ]:
# Run evaluation on eval dataset
eval_results = trainer.evaluate()

print("\n" + "="*80)
print("📊 EVALUATION RESULTS")
print("="*80 + "\n")

for key, value in eval_results.items():
    print(f"   {key}: {value}")

print("\n" + "="*80)
print("✅ FINE-TUNING PIPELINE COMPLETE!")
print("="*80)

## 🧪 Step 2.8: Test Fine-tuned Model on Sample Video

In [ ]:
# Test the fine-tuned model on a sample
test_sample = eval_dataset[0]

# Prepare input
messages = test_sample
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
images = [item["image"] for item in messages[1]["content"] if item["type"] == "image"]

# Process inputs
inputs = processor(
    text=[text],
    images=images,
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate
print("\n🤖 Testing fine-tuned model on sample...")
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False
    )

# Decode output
generated_ids_trimmed = [
    out_ids[len(in_ids):]
    for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(f"\n{'='*80}")
print(f"🧪 FINE-TUNED MODEL TEST")
print(f"{'='*80}\n")
print(f"❓ Question: {test_sample[1]['content'][-1]['text'][:200]}...")
print(f"\n🤖 Model Answer: {output_text}")
print(f"\n✅ Ground Truth: {test_sample[2]['content'][0]['text']}")
print(f"\n{'='*80}\n")